# EDA and model visualizations

The original comparison chart uses manually entered metrics, and some differ from the final report. The EDA cells can fall back to simulated inputs when files are missing. Verify the inputs before regenerating figures for submission.

Run from the repository root.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from math import pi

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style="whitegrid", palette="muted")

# ==========================================
# Model metrics used in the comparison plots
# ==========================================
raw_results = [
    # TextRCNN models
    {"Model": "TextRCNN + SGNS", "accuracy": 0.642, "macro_f1": 0.543, "precision": 0.544, "recall": 0.542},
    {"Model": "TextRCNN + GloVe", "accuracy": 0.603, "macro_f1": 0.518, "precision": 0.525, "recall": 0.516},
    {"Model": "TextRCNN + FastText", "accuracy": 0.6245, "macro_f1": 0.5285, "precision": 0.5251, "recall": 0.5389},
    
    # DPCNN models
    {"Model": "DPCNN + SGNS",    "accuracy": 0.642, "macro_f1": 0.527, "precision": 0.540, "recall": 0.520},
    {"Model": "DPCNN + GloVe",   "accuracy": 0.634, "macro_f1": 0.532, "precision": 0.536, "recall": 0.533},
    {"Model": "DPCNN + FastText", "accuracy": 0.6485, "macro_f1": 0.5393, "precision": 0.5480, "recall": 0.5359},
    
    # Logistic regression baseline 
    {"Model": "Baseline (LR)",   "accuracy": 0.580, "macro_f1": 0.470, "precision": 0.480, "recall": 0.460}
]

df_results = pd.DataFrame(raw_results)
df_results = df_results.sort_values(by="macro_f1", ascending=False) # Sort by macro-F1 in descending order.
print("Comparison metrics for seven models:")
display(df_results)

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from wordcloud import WordCloud

# Set the plotting font and style.
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style="whitegrid", palette="muted")

# Set the path to the archived input data.
DATA_DIR = Path.cwd() / 'data_64_16_20'

# ==========================================
# Figure 1. Rating distribution
# ==========================================
try:
    # Try to load ratings from the project data.
    with open(DATA_DIR / 'manifest.json') as f:
        manifest = json.load(f)
    train_file = DATA_DIR / manifest['split_info']['train']['file']
    ratings = [json.loads(line)['rating'] for line in train_file.read_text().splitlines()]
except Exception as e:
    print(f"⚠️ Could not read project data ({e}). Plotting a simulated distribution.")
    # Simulate a distribution with many ratings of 1 and 10.
    ratings = [1]*150 + [2]*80 + [3]*60 + [4]*40 + [5]*50 + [6]*60 + [7]*90 + [8]*200 + [9]*400 + [10]*800

plt.figure(figsize=(10, 5))
sns.countplot(x=ratings, palette="viridis")
plt.title('EDA 1: Rating Distribution in Training Set', fontsize=16)
plt.xlabel('Rating Score (1-10)', fontsize=12)
plt.ylabel('Number of Reviews', fontsize=12)

# Label the bars with their counts.
ax = plt.gca()
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 5), textcoords='offset points', fontsize=9)

plt.tight_layout()
plt.savefig('fig1_eda_distribution.png', dpi=300)
plt.show()


# ==========================================
# Figure 2. Positive and negative review word clouds
# ==========================================
try:
    # Try to extract positive and negative reviews from the project data.
    texts = []
    for line in train_file.read_text().splitlines():
        row = json.loads(line)
        texts.append((row['rating'], " ".join(row['filtered_words'])))
    
    low_texts = " ".join([t for r, t in texts if r <= 3])
    high_texts = " ".join([t for r, t in texts if r >= 9])
    
    if not low_texts or not high_texts:
        raise ValueError("The project review text is empty.")
except Exception as e:
    print(f"⚠️ Could not read review text ({e}). Plotting simulated words.")
    # Use simulated examples of frequent words.
    low_texts = "side effect pain bad sick headache terrible awful worse nausea doctor stop terrible " * 100
    high_texts = "life great perfect help work good amazing wonderful miracle feel better excellent " * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Negative review word cloud (red)
wc_low = WordCloud(width=800, height=400, background_color='white', 
                   colormap='Reds', max_words=100).generate(low_texts)
axes[0].imshow(wc_low, interpolation='bilinear')
axes[0].axis('off')
axes[0].set_title('EDA 2: Word Cloud for Negative Reviews (Low)', fontsize=16, color='darkred')

# Positive review word cloud (green)
wc_high = WordCloud(width=800, height=400, background_color='white', 
                    colormap='Greens', max_words=100).generate(high_texts)
axes[1].imshow(wc_high, interpolation='bilinear')
axes[1].axis('off')
axes[1].set_title('EDA 2: Word Cloud for Positive Reviews (Excellent)', fontsize=16, color='darkgreen')

plt.tight_layout()
plt.savefig('fig2_eda_wordcloud.png', dpi=300)
plt.show()

In [ ]:
# ==========================================
# Figure 3. Grouped model comparison
# ==========================================
# Select the metrics to display.
df_melted = df_results.melt(id_vars='Model', value_vars=['accuracy', 'macro_f1'], var_name='Metric', value_name='Score')

plt.figure(figsize=(14, 6))
sns.barplot(data=df_melted, x='Model', y='Score', hue='Metric', palette='Set2')
plt.axhline(y=0.470, color='red', linestyle='--', label='LR Baseline (F1)')
plt.title('Result 1: Model Performance Comparison (7 Models)', fontsize=16)
plt.ylim(0, 0.8)
plt.xticks(rotation=15, fontsize=11)
plt.legend(loc='upper right')
plt.tight_layout()
plt.savefig('fig3_result_model_comparison.png', dpi=300)
plt.show()

# ==========================================
# Figure 4. Radar chart for the top three models and baseline
# ==========================================
top3 = df_results.head(3)
df_radar = pd.concat([top3, df_results[df_results['Model'] == 'Baseline (LR)']])

categories = ['accuracy', 'macro_f1', 'precision', 'recall']
N = len(categories)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
colors = ['blue', 'orange', 'green', 'red']
for idx, (_, row) in enumerate(df_radar.iterrows()):
    values = row[categories].values.tolist()
    values += values[:1]
    ax.plot(angles, values, linewidth=2, label=row['Model'], color=colors[idx])
    ax.fill(angles, values, alpha=0.1, color=colors[idx])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(['Accuracy', 'Macro-F1', 'Precision', 'Recall'], fontsize=12)
ax.set_ylim(0, 0.8)
plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.title('Result 2: Radar Chart of Top 3 Models vs Baseline', fontsize=15)
plt.savefig('fig4_result_radar_chart.png', dpi=300)
plt.show()

# ==========================================
# Figure 5. Learning curves
# ==========================================
plt.figure(figsize=(10, 6))

# DPCNN + GloVe
plt.plot([1,2,3,4,5], [0.4433, 0.5098, 0.496, 0.4997, 0.5318], marker='o', label='DPCNN + GloVe', linewidth=2)
# DPCNN + SGNS
plt.plot([1,2,3,4,5,6], [0.4308, 0.5137, 0.4821, 0.4856, 0.5272, 0.5134], marker='s', label='DPCNN + SGNS', linewidth=2, color='purple')
# TextRCNN + GloVe
textrcnn_glove_f1 = [0.4158, 0.4555, 0.4599, 0.4523, 0.48, 0.4867, 0.5009, 0.4971, 0.5012, 0.5041, 
                     0.5104, 0.5058, 0.5147, 0.5168, 0.5062, 0.5112, 0.5134, 0.5106, 0.5177, 0.5181]
plt.plot(range(1, 21), textrcnn_glove_f1, marker='^', label='TextRCNN + GloVe (20 ep)', linewidth=2, color='green', alpha=0.7)

plt.title('Result 3: Validation Macro-F1 over Epochs (Top Models)', fontsize=15)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Macro-F1 Score', fontsize=12)
plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('fig5_result_learning_curve.png', dpi=300)
plt.show()